## 1. Data Loading and Environment Setup

This section initialises the analytical environment and loads the processed macroeconomic dataset. A dynamic path detection mechanism ensures portability across different computing environments, facilitating reproducible research and collaborative workflows.

In [1]:
# Step 1: Setup and Automatic Dataset Loading



from pathlib import Path



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



# Plotting & Pandas display settings

plt.style.use("seaborn-v0_8")

sns.set(font_scale=1.1)



pd.set_option("display.max_columns", 100)

pd.set_option("display.width", 140)



# Find project root by walking up until a folder containing `data/` is found

def find_project_root(marker: str = "data") -> Path:

    cwd = Path.cwd()

    for path in (cwd, *cwd.parents):

        if (path / marker).exists():

            return path

    raise FileNotFoundError(f"Could not find project root containing '{marker}' directory from {cwd}")



PROJECT_ROOT = find_project_root()



# Construct path to processed dataset inside repo

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final_merged_dataset.csv"



# Load dataset

df = pd.read_csv(DATA_PATH)



print(f"Loaded dataset from: {DATA_PATH}")

print("Shape:", df.shape)



df.head()


Loaded dataset from: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\data\processed\final_merged_dataset.csv
Shape: (338, 8)


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil,Bank Rate
0,1.7,1997-06-01,1997-06,1.0,1.6446,7.3,9.3,5.9375
1,2.0,1997-07-01,1997-07,0.6,1.6702,7.1,14.0,5.9375
2,2.0,1997-08-01,1997-08,0.9,1.6034,6.8,14.1,5.9375
3,1.8,1997-09-01,1997-09,0.8,1.6015,6.7,11.2,5.9375
4,1.9,1997-10-01,1997-10,1.0,1.6329,6.6,8.5,5.9375


## 2. Feature Engineering Workflow

This section creates the complete feature set for multi-horizon inflation forecasting:

### Step 1: Create Future Target Variables
- **cpi_t1**: CPI YoY one month ahead
- **cpi_t3**: CPI YoY three months ahead  
- **cpi_t5**: CPI YoY five months ahead

### Step 2: Create Lag Features (Past Values)
Lag features capture historical patterns:
- **CPI lags**: t-1, t-3, t-6
- **Petrol/Oil lags**: t-1
- **Exchange rate lags**: t-1
- **Bank rate lags**: t-1
- **GVA growth lags**: t-1
- **Unemployment lags**: t-1

### Step 3: Clean Missing Values
Remove rows with NaN values introduced by shift operations (future targets at end, lags at beginning).

In [2]:
# Step 2: Complete Feature Engineering Pipeline

# Define column name mappings for cleaner code
cpi_col     = "CPI ANNUAL RATE 00: ALL ITEMS 2015=100"
petrol_col  = "RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil"
bank_col    = "Bank Rate"
fx_col      = "Exchange_USD"
unemp_col   = "Unemployment rate (aged 16 and over, seasonally adjusted): %"
gva_col     = "Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"
date_col    = "Date"

# Create simplified column names for easier use
df = df.rename(columns={
    cpi_col: "cpi_yoy",
    petrol_col: "petrol_yoy",
    bank_col: "bank_rate",
    fx_col: "fx_usd",
    unemp_col: "unemp_rate",
    gva_col: "gva_growth",
    date_col: "date"
})

# Convert date to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort by date to ensure correct ordering
df = df.sort_values("date").reset_index(drop=True)

print("="*80)
print("STEP 1: CREATE FUTURE TARGETS")
print("="*80)

# Create future target variables (negative shift = look ahead)
df["cpi_t1"] = df["cpi_yoy"].shift(-1)  # 1 month ahead
df["cpi_t3"] = df["cpi_yoy"].shift(-3)  # 3 months ahead
df["cpi_t5"] = df["cpi_yoy"].shift(-5)  # 5 months ahead

# Drop rows where targets are missing (last 5 rows)
df_targets = df.dropna(subset=["cpi_t1", "cpi_t3", "cpi_t5"]).copy()

print(f"Original shape: {df.shape}")
print(f"After dropping missing targets: {df_targets.shape}")
print(f"Rows removed (end of dataset): {df.shape[0] - df_targets.shape[0]}")

print("\n" + "="*80)
print("STEP 2: CREATE LAG FEATURES")
print("="*80)

# Create lag features (positive shift = look back)
# CPI lags: 1, 3, 6 months
df_targets["cpi_yoy_l1"] = df_targets["cpi_yoy"].shift(1)
df_targets["cpi_yoy_l3"] = df_targets["cpi_yoy"].shift(3)
df_targets["cpi_yoy_l6"] = df_targets["cpi_yoy"].shift(6)

# Other variable lags: 1 month only
df_targets["petrol_yoy_l1"] = df_targets["petrol_yoy"].shift(1)
df_targets["fx_usd_l1"] = df_targets["fx_usd"].shift(1)
df_targets["bank_rate_l1"] = df_targets["bank_rate"].shift(1)
df_targets["gva_growth_l1"] = df_targets["gva_growth"].shift(1)
df_targets["unemp_rate_l1"] = df_targets["unemp_rate"].shift(1)

# Drop rows where lag features are missing (first 6 rows due to cpi_yoy_l6)
df_final = df_targets.dropna().reset_index(drop=True)

print(f"After creating lags: {df_targets.shape}")
print(f"After dropping missing lags: {df_final.shape}")
print(f"Rows removed (start of dataset): {df_targets.shape[0] - df_final.shape[0]}")

print("\n" + "="*80)
print("STEP 3: REORDER COLUMNS")
print("="*80)

# Define column order: date, current values, lag features, targets
column_order = [
    "date",
    # Current values
    "cpi_yoy", "petrol_yoy", "bank_rate", "fx_usd", "gva_growth", "unemp_rate",
    # Lag features
    "cpi_yoy_l1", "cpi_yoy_l3", "cpi_yoy_l6",
    "petrol_yoy_l1", "fx_usd_l1", "bank_rate_l1", "gva_growth_l1", "unemp_rate_l1",
    # Future targets
    "cpi_t1", "cpi_t3", "cpi_t5"
]

# Remove YearMonth column if it exists
if "YearMonth" in df_final.columns:
    df_final = df_final.drop(columns=["YearMonth"])

# Reorder columns
df_final = df_final[column_order]

print("Final column order:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"Shape: {df_final.shape}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")
print(f"Total usable observations: {len(df_final)}")

print("\n" + "="*80)
print("FIRST 5 ROWS (to verify structure)")
print("="*80)
display(df_final.head())

print("\n" + "="*80)
print("LAST 5 ROWS (to verify targets)")
print("="*80)
display(df_final.tail())

# Update df to be the final engineered dataset
df = df_final.copy()

STEP 1: CREATE FUTURE TARGETS
Original shape: (338, 11)
After dropping missing targets: (333, 11)
Rows removed (end of dataset): 5

STEP 2: CREATE LAG FEATURES
After creating lags: (333, 19)
After dropping missing lags: (327, 19)
Rows removed (start of dataset): 6

STEP 3: REORDER COLUMNS
Final column order:
   1. date
   2. cpi_yoy
   3. petrol_yoy
   4. bank_rate
   5. fx_usd
   6. gva_growth
   7. unemp_rate
   8. cpi_yoy_l1
   9. cpi_yoy_l3
  10. cpi_yoy_l6
  11. petrol_yoy_l1
  12. fx_usd_l1
  13. bank_rate_l1
  14. gva_growth_l1
  15. unemp_rate_l1
  16. cpi_t1
  17. cpi_t3
  18. cpi_t5

FINAL DATASET SUMMARY
Shape: (327, 18)
Date range: 1997-12-01 00:00:00 to 2025-02-01 00:00:00
Total usable observations: 327

FIRST 5 ROWS (to verify structure)


,date,cpi_yoy,petrol_yoy,bank_rate,fx_usd,gva_growth,unemp_rate,cpi_yoy_l1,cpi_yoy_l3,cpi_yoy_l6,petrol_yoy_l1,fx_usd_l1,bank_rate_l1,gva_growth_l1,unemp_rate_l1,cpi_t1,cpi_t3,cpi_t5
0,1997-12-01,1.7,3.4,5.9375,1.6597,1.4,6.4,1.9,1.8,1.7,7.5,1.6890,5.9375,1.0,6.5,1.5,1.7,2.0
1,1998-01-01,1.5,3.2,7.2500,1.6353,1.4,6.4,1.7,1.9,2.0,3.4,1.6597,5.9375,1.4,6.4,1.6,1.8,1.7
2,1998-02-01,1.6,2.9,7.2500,1.6407,1.4,6.4,1.5,1.9,2.0,3.2,1.6353,7.2500,1.4,6.4,1.7,2.0,1.4
3,1998-03-01,1.7,3.3,7.2500,1.6620,0.8,6.3,1.6,1.7,1.8,2.9,1.6407,7.2500,1.4,6.4,1.8,1.7,1.3
4,1998-04-01,1.8,10.0,7.2500,1.6733,0.9,6.3,1.7,1.5,1.9,3.3,1.6620,7.2500,0.8,6.3,2.0,1.4,1.4



LAST 5 ROWS (to verify targets)


,date,cpi_yoy,petrol_yoy,bank_rate,fx_usd,gva_growth,unemp_rate,cpi_yoy_l1,cpi_yoy_l3,cpi_yoy_l6,petrol_yoy_l1,fx_usd_l1,bank_rate_l1,gva_growth_l1,unemp_rate_l1,cpi_t1,cpi_t3,cpi_t5
322,2024-10-01,2.3,-14.2,5.25,1.3045,0.2,4.4,1.7,2.2,2.3,-11.6,1.3217,5.25,0.2,4.3,2.6,3.0,2.6
323,2024-11-01,2.6,-12.2,5.25,1.2750,0.1,4.4,2.3,2.2,2.0,-14.2,1.3045,5.25,0.2,4.4,2.5,2.8,3.5
324,2024-12-01,2.5,-6.0,5.25,1.2647,0.2,4.4,2.6,1.7,2.0,-12.2,1.2750,5.25,0.1,4.4,3.0,2.6,3.4
325,2025-01-01,3.0,-2.8,4.50,1.2348,0.3,4.4,2.5,2.3,2.2,-6.0,1.2647,5.25,0.2,4.4,2.8,3.5,3.6
326,2025-02-01,2.8,-2.4,4.75,1.2545,0.6,4.5,3.0,2.6,2.2,-2.8,1.2348,4.50,0.3,4.4,2.6,3.4,3.8


In [ ]:
# Step 3: Align with config + pipeline preprocessors
import sys, yaml
sys.path.append(str(PROJECT_ROOT))
from src.data.preprocessor import temporal_splits, scale_features

CONFIG_PATH = PROJECT_ROOT / "configs" / "hybrid.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text())
feature_cols = config["features"]["feature_cols"]
target_cols = config["features"]["target_cols"]
expected_cols = ["date"] + feature_cols + target_cols

missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in engineered dataframe: {missing}")

# Reorder and trim to expected columns
df = df[expected_cols].copy()

train_end = pd.to_datetime(config["data"]["train_end"])
val_end = pd.to_datetime(config["data"]["val_end"])

train_mask, val_mask, test_mask = temporal_splits(
    df,
    train_end=train_end,
    val_end=val_end,
)

X_all = df[feature_cols]
y_all = df[target_cols]

X_train, X_val, X_test, y_train, y_val, y_test, scaler_X, scaler_y = scale_features(
    X_all,
    y_all,
    train_mask,
    val_mask,
    test_mask,
)

print("Config-loaded feature columns:", feature_cols)
print("Target columns:", target_cols)
print("Split sizes (train/val/test):", train_mask.sum(), val_mask.sum(), test_mask.sum())
print(
    "Scaled shapes:",
    "X_train", X_train.shape,
    "X_val", X_val.shape,
    "X_test", X_test.shape,
    "y_train", y_train.shape,
    "y_val", y_val.shape,
    "y_test", y_test.shape,
)

Config-loaded feature columns: ['cpi_yoy', 'petrol_yoy', 'bank_rate', 'fx_usd', 'gva_growth', 'unemp_rate', 'cpi_yoy_l1', 'cpi_yoy_l3', 'cpi_yoy_l6', 'petrol_yoy_l1', 'fx_usd_l1', 'bank_rate_l1', 'gva_growth_l1', 'unemp_rate_l1']
Target columns: ['cpi_t1', 'cpi_t3', 'cpi_t5']
Split sizes (train/val/test): 241 48 38
Scaled shapes: X_train (241, 14) X_val (48, 14) X_test (38, 14) y_train (241, 3) y_val (48, 3) y_test (38, 3)


## Checking the dataset start and end dates + saving a model-ready CSV file

Before moving forward, it is important to confirm the time coverage of the dataset.
This helps ensure there are no unexpected gaps or missing years.

In this step, I will:

1. Convert the `date` column to a proper datetime type (if needed).
2. Display the earliest and latest dates in the dataset.
3. Save the cleaned and feature-engineered dataset as  
   **`model_ready_dataset.csv`** for modelling.

This helps maintain a clear workflow and keeps my modelling files organised.


In [4]:
# Ensure date column is datetime (safe even if already datetime)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Check dataset start and end dates
start_date = df["date"].min()
end_date = df["date"].max()

print("Dataset start date:", start_date)
print("Dataset end date:  ", end_date)

# Save the final dataset (after feature engineering)
df.to_csv("model_ready_dataset.csv", index=False)

print("\nSaved file as: model_ready_dataset.csv")


Dataset start date: 1997-12-01 00:00:00
Dataset end date:   2025-02-01 00:00:00

Saved file as: model_ready_dataset.csv
